<a href="https://colab.research.google.com/github/PollyIva/DL_in_CV/blob/PollyIva-patch-1/_deepsort_project_ipynb%206.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 1. Установка библиотек и Импорт



In [ ]:
!nvidia-smi

/bin/bash: line 1: nvidia-smi: command not found


In [ ]:
!pip install ultralytics
!pip install torchreid
!pip install motmetrics
!pip install gdown
# !pip install onnxruntime --quiet

!git clone https://github.com/nwojke/deep_sort.git
%cd deep_sort
!pip install -r requirements.txt
%cd ..

fatal: destination path 'deep_sort' already exists and is not an empty directory.
/content/deep_sort
/content


In [ ]:
import os
import sys
sys.path.append("deep_sort")
print("DeepSORT path added")
import cv2
import torch
import numpy as np
import time

import torchreid
import torchvision.transforms as T
from PIL import Image
import csv
import json
import hashlib
sys.path.append('./deep_sort')

from deep_sort import nn_matching
from deep_sort.detection import Detection
from deep_sort.tracker import Tracker
from ultralytics import YOLO
# import onnxruntime as ort

import pandas as pd
import shutil
import tempfile
# import kagglehub
!git clone https://github.com/JonathonLuiten/TrackEval.git

import sys
sys.path.append("TrackEval")

import trackeval
print(trackeval.__file__)

from trackeval import Evaluator
from trackeval.datasets import MotChallenge2DBox
from trackeval.metrics import HOTA, CLEAR, Identity

DeepSORT path added
fatal: destination path 'TrackEval' already exists and is not an empty directory.
/content/TrackEval/trackeval/__init__.py


In [ ]:
np.float = float
np.int = int

In [ ]:
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", DEVICE)

PyTorch: 2.11.0+cpu
CUDA available: False
Using device: cpu


# 2. Настройки проекта

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
BASE_DIR = "/content/drive/MyDrive/DeepSORT"

folders = [
    "videos",
    "results",
    "overlays",
    "metrics",
    "configs"
]

# Создаем основную папку
os.makedirs(BASE_DIR, exist_ok=True)

# Создаем подпапки
for folder in folders:
    os.makedirs(os.path.join(BASE_DIR, folder), exist_ok=True)

print("Folders created")

Folders created


In [ ]:
VIDEOS_DIR = os.path.join(BASE_DIR, "videos")
RESULTS_DIR = os.path.join(BASE_DIR, "results")
OVERLAYS_DIR = os.path.join(BASE_DIR, "overlays")
METRICS_DIR = os.path.join(BASE_DIR, "metrics")
CONFIGS_DIR = os.path.join(BASE_DIR, "configs")

В диск были скопированы датасеты
https://www.kaggle.com/datasets/takshmandar/mot16-dataset
https://www.kaggle.com/datasets/mdraselsarker/mot15-challenge-dataset?resource=download

In [ ]:
DATA_DIR = os.path.join(BASE_DIR, "MOT_datasets")
os.makedirs(DATA_DIR, exist_ok=True)

In [ ]:
SEQUENCES = sorted([
    d for d in os.listdir(DATA_DIR)
    if os.path.isdir(os.path.join(DATA_DIR, d))
])

print(SEQUENCES)

['KITTI-17', 'MOT16-09', 'MOT16-11', 'PETS09-S2L1', 'TUD-Campus', 'TUD-Stadtmitte']


In [ ]:
CACHE_DIR = os.path.join(BASE_DIR, "cache")
os.makedirs(CACHE_DIR, exist_ok=True)


def make_key(*args):
    raw = "_".join(map(str, args))
    return hashlib.md5(raw.encode()).hexdigest()


def load_cache(key):
    path = os.path.join(CACHE_DIR, f"{key}.json")
    if os.path.exists(path):
        with open(path) as f:
            return json.load(f)
    return None


def save_cache(key, data):
    path = os.path.join(CACHE_DIR, f"{key}.json")
    with open(path, "w") as f:
        json.dump(data, f, indent=2)

таблица результатов

In [ ]:
os.makedirs("/content/drive/MyDrive/DeepSORT", exist_ok=True)

output_csv = os.path.join(BASE_DIR, "experiments.csv")
os.makedirs(os.path.dirname(output_csv), exist_ok=True)

fieldnames = [
    "sequence", "detector", "reid",
    "conf", "cosine_distance", "nn_budget",
    "fps", "MOTA", "IDF1",
    "HOTA"
]

file_exists = os.path.exists(output_csv)

#3. Detectors

In [ ]:
#Базовый класс детектора
class Detector:
    def detect(self, frame):
        raise NotImplementedError

In [ ]:
class YOLOv5Detector(Detector):
    def __init__(self, model_name="yolov5n.pt", conf=0.5, device=DEVICE):
        self.model = YOLO(model_name)  # ultralytics wrapper
        self.conf = conf
        self.device = device

    def detect(self, frame):
        result = self.model(frame, conf=self.conf, device=self.device, verbose=False)[0]

        detections = []
        for box in result.boxes:
            if int(box.cls[0]) != 0:  # person class
                continue

            x1, y1, x2, y2 = box.xyxy[0].cpu().numpy()

            detections.append([
                int(x1), int(y1), int(x2), int(y2), float(box.conf[0])
            ])

        return detections

In [ ]:
#  Фабрика детекторов

def create_detector(name, conf=0.5):
    if name == "yolov5n":
        return YOLOv5Detector("yolov5n.pt", conf=conf)
    elif name == "yolov5m":
        return YOLOv5Detector("yolov5m.pt", conf=conf)
    elif name == "yolov5l6":
        return YOLOv5Detector("yolov5l6.pt", conf=conf)
    else:
        raise ValueError(f"Unknown detector: {name}")

In [ ]:
detector_name = "yolov5l6"
detector = create_detector(detector_name)
print(
    "Loaded detector:",
    detector_name
)

PRO TIP 💡 Replace 'model=yolov5l6.pt' with new 'model=yolov5l6u.pt'.
YOLOv5 'u' models are trained with https://github.com/ultralytics/ultralytics and feature improved performance vs standard YOLOv5 models trained with https://github.com/ultralytics/yolov5.
Loaded detector: yolov5l6


# 4. ReID


In [ ]:
class ReIDModel:
    def get_embedding(self, image):
        raise NotImplementedError

In [ ]:
class TorchReID(ReIDModel):
    def __init__(self, model_name, device=DEVICE):
        self.device = device
        self.model = torchreid.models.build_model(
            name=model_name,
            num_classes=1000,
            pretrained=True
        )
        self.model.to(device)
        self.model.eval()

        self.transform = T.Compose([
            T.ToPILImage(),
            T.Resize((256, 128)),
            T.ToTensor(),
            T.Normalize(
                mean=[0.485, 0.456, 0.406],
                std=[0.229, 0.224, 0.225]
            )
        ])

    @torch.no_grad()
    def get_embedding(self, image):
        """Остается для обратной совместимости"""
        img = self.transform(image).unsqueeze(0).to(self.device)
        feature = self.model(img).cpu().numpy()[0]
        feature /= np.linalg.norm(feature)
        return feature

    @torch.no_grad()
    def get_embeddings_batch(self, images):
        """Батчевая обработка списка изображений"""
        if not images:
            return []

        # Трансформируем каждый кроп и собираем в единый батч-тензор
        tensors = [self.transform(img) for img in images]
        batch = torch.stack(tensors).to(self.device)  # Форма: [N, C, H, W]

        # Прогоняем весь батч через модель за один пасс
        features = self.model(batch).cpu().numpy()  # Форма: [N, Features]

        # L2 нормализация по строкам (для каждого эмбеддинга отдельно)
        norms = np.linalg.norm(features, axis=1, keepdims=True)
        features /= np.zeros_like(norms) + norms + 1e-8  # Защита от деления на 0

        return features

In [ ]:
def create_reid(name):

    if name == "osnet_x0_25":
        return TorchReID(
            "osnet_x0_25"
        )

    elif name == "osnet_x1_0":
        return TorchReID(
            "osnet_x1_0"
        )

    elif name == "resnet50":
        return TorchReID(
            "resnet50"
        )

    else:
        raise ValueError(
            f"Unknown REID model: {name}"
        )

In [ ]:
reid_name = "osnet_x0_25"

reid = create_reid(reid_name)

print(
    "Loaded REID:",
    reid_name
)

Successfully loaded imagenet pretrained weights from "/root/.cache/torch/checkpoints/osnet_x0_25_imagenet.pth"
Loaded REID: osnet_x0_25


# 5. DeepSORT.

In [ ]:
class DeepSORTWrapper:
    def __init__(self, max_cosine_distance=0.3, nn_budget=100):
        self.metric = nn_matching.NearestNeighborDistanceMetric(
            "cosine",
            max_cosine_distance,
            nn_budget
        )
        self.tracker = Tracker(self.metric)

    def update(self, frame, detections, reid):
        """
        detections: [[x1, y1, x2, y2, confidence], ...]
        """
        valid_detections = []
        crops = []

        # Шаг 1: Только собираем кропы и фильтруем пустые детекции
        for det in detections:
            x1, y1, x2, y2, conf = det

            # Ограничиваем координаты границами кадра, чтобы избежать пустых или искаженных кропов
            h, w = frame.shape[:2]
            x1, y1 = max(0, int(x1)), max(0, int(y1))
            x2, y2 = min(w, int(x2)), min(h, int(y2))

            crop = frame[y1:y2, x1:x2]
            if crop.size == 0:
                continue

            crops.append(crop)
            valid_detections.append([x1, y1, x2, y2, conf])

        # Шаг 2: Извлекаем эмбеддинги ОДНИМ батчем (вызов GPU происходит один раз)
        features = reid.get_embeddings_batch(crops)

        # Шаг 3: Формируем объекты Detection для DeepSORT
        ds_detections = []
        for det, feature in zip(valid_detections, features):
            x1, y1, x2, y2, conf = det

            # Переводим в формат tlwh (top_left_x, top_left_y, width, height)
            bbox = [x1, y1, x2 - x1, y2 - y1]

            ds_detections.append(
                Detection(bbox, conf, feature)
            )

        # Шаг 4: Стандартный пайплайн трекера
        self.tracker.predict()
        self.tracker.update(ds_detections)

        results = []
        for track in self.tracker.tracks:
            if not track.is_confirmed():
                continue
            if track.time_since_update > 1:
                continue

            bbox = track.to_tlbr()
            results.append({
                "id": track.track_id,
                "bbox": bbox
            })

        return results

In [ ]:
tracker = DeepSORTWrapper(
    max_cosine_distance=0.3,
    nn_budget=100
)
detector = create_detector("yolov5n")
reid = create_reid("osnet_x0_25")

PRO TIP 💡 Replace 'model=yolov5n.pt' with new 'model=yolov5nu.pt'.
YOLOv5 'u' models are trained with https://github.com/ultralytics/ultralytics and feature improved performance vs standard YOLOv5 models trained with https://github.com/ultralytics/yolov5.
Successfully loaded imagenet pretrained weights from "/root/.cache/torch/checkpoints/osnet_x0_25_imagenet.pth"


# 6. сохранение результатов и Визуализация



In [ ]:
class MOTWriter:
    def __init__(self, filename):
        self.file = open(filename, "w")


    def write(self, frame_id, track_id, bbox):
        x1, y1, x2, y2 = bbox

        width = x2 - x1
        height = y2 - y1

        line = (
            f"{frame_id},{track_id},"
            f"{x1:.2f},{y1:.2f},"
            f"{width:.2f},{height:.2f},"
            "1,-1,-1,-1\n"
        )

        self.file.write(line)


    def close(self):
        self.file.close()

In [ ]:
def draw_tracks(frame, tracks):

    for track in tracks:

        x1, y1, x2, y2 = map(int, track["bbox"])

        track_id = track["id"]

        cv2.rectangle(
            frame,
            (x1, y1),
            (x2, y2),
            (0, 255, 0),
            2
        )

        cv2.putText(
            frame,
            f"ID {track_id}",
            (x1, y1 - 10),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.7,
            (255, 255, 255),
            2
        )

    return frame

In [ ]:
def run_sequence(
    img_dir,
    output_video,
    mot_path,
    detector,
    reid,
    tracker,
    fps=30
):

    frames = sorted(os.listdir(img_dir))

    first_frame = cv2.imread(os.path.join(img_dir, frames[0]))
    h, w = first_frame.shape[:2]

    writer = cv2.VideoWriter(
        output_video,
        cv2.VideoWriter_fourcc(*"mp4v"),
        fps,
        (w, h)
    )

    mot_writer = MOTWriter(mot_path)

    total_time = 0

    for i, frame_name in enumerate(frames, start=1):

        frame_path = os.path.join(img_dir, frame_name)
        frame = cv2.imread(frame_path)

        if frame is None:
            continue

        start = time.time()

        detections = detector.detect(frame)
        tracks = tracker.update(frame, detections, reid)

        elapsed = time.time() - start
        total_time += elapsed

        fps_now = 1 / elapsed if elapsed > 0 else 0

        cv2.putText(frame, f"FPS: {fps_now:.1f}", (20, 40),
                    cv2.FONT_HERSHEY_SIMPLEX, 1, (0,0,255), 2)

        frame = draw_tracks(frame, tracks)

        for t in tracks:
            mot_writer.write(i, t["id"], t["bbox"])

        writer.write(frame)

        if i % 50 == 0:
            print(f"Frame {i}/{len(frames)} FPS={fps_now:.2f}")

    writer.release()
    mot_writer.close()

    avg_fps = len(frames) / total_time if total_time > 0 else 0

    print("DONE")
    print("AVG FPS:", avg_fps)

    return avg_fps

10. Grid Search

11. Анализ результатов

In [ ]:
detectors = [
    "yolov5n",
    # "yolov5m",
    # "yolov5l6",
]


reids = [
    "osnet_x0_25",
    # "osnet_x1_0",
    # "resnet50"
]


confidences = [
    # 0.3,
    0.5,
    # 0.7
]


cosine_distances = [
#     0.2,
#     0.3,
    0.5
]


nn_budgets = [
    50,
    # 100
]

In [ ]:
def compute_metrics(sequence_path, pred_file):
    print(f"Evaluating sequence: {sequence_path}")
    seq_name = os.path.basename(sequence_path)

    tmp = tempfile.mkdtemp()
    gt_root = os.path.join(tmp, "gt")
    tracker_root = os.path.join(tmp, "trackers")


    # ВАЖНО: Мы жестко фиксируем BENCHMARK = "MOT16" для ВСЕХ папок.
    # Это заставит TrackEval использовать один и тот же предсказуемый шаблон путей.
    benchmark = "MOT16"

    # Создаем правильную структуру папок, которую ожидает MOT16 при SKIP_SPLIT_FOL=True
    gt_seq_dir = os.path.join(gt_root, seq_name)
    os.makedirs(gt_seq_dir, exist_ok=True)

    # Копируем файл конфигурации seqinfo.ini
    src_ini = os.path.join(sequence_path, "seqinfo.ini")
    dst_ini = os.path.join(gt_seq_dir, "seqinfo.ini")
    shutil.copy(src_ini, dst_ini)

    # Создаём папку gt и копируем туда gt.txt
    gt_gt_dir = os.path.join(gt_seq_dir, "gt")
    os.makedirs(gt_gt_dir, exist_ok=True)
    shutil.copy(
        os.path.join(sequence_path, "gt", "gt.txt"),
        os.path.join(gt_gt_dir, "gt.txt")
    )

    # Результаты трекера складываем по структуре, которую TrackEval/MOT16 считает стандартной
    tracker_name = "DeepSORT"

    # Для MOT16 + SKIP_SPLIT_FOL=True библиотека ищет файл в: {TRACKERS_FOLDER}/{tracker_name}/data/{seq_name}.txt
    tracker_dir = os.path.join(tracker_root, tracker_name, "data")
    os.makedirs(tracker_dir, exist_ok=True)
    shutil.copy(pred_file, os.path.join(tracker_dir, f"{seq_name}.txt"))

    # Генерируем seqmap-файл
    seqmap_dir = os.path.join(tmp, "seqmaps")
    os.makedirs(seqmap_dir, exist_ok=True)
    seqmap_path = os.path.join(seqmap_dir, f"{benchmark}-train.txt")

    with open(seqmap_path, "w") as f:
        f.write("name\n")
        f.write(seq_name + "\n")

    # Настройки самого оценщика
    eval_config = Evaluator.get_default_eval_config()
    dataset_config = MotChallenge2DBox.get_default_dataset_config()

    dataset_config["GT_FOLDER"] = gt_root
    dataset_config["TRACKERS_FOLDER"] = tracker_root
    dataset_config["BENCHMARK"] = benchmark
    dataset_config["SKIP_SPLIT_FOL"] = True  # Игнорируем деление на train/test подпапки
    dataset_config["SEQMAP_FOLDER"] = seqmap_dir
    dataset_config["TRACKERS_TO_EVAL"] = [tracker_name]
    dataset_config["DO_PREPROC"] = False
    # Дополнительно настраиваем маску путей для GT, так как у нас плоская структура
    dataset_config["GT_LOC_FORMAT"] = "{gt_folder}/{seq}/gt/gt.txt"

    evaluator = Evaluator(eval_config)
    dataset = MotChallenge2DBox(dataset_config)

    metrics = [HOTA(), CLEAR(), Identity()]
    results, _ = evaluator.evaluate([dataset], metrics)

    res_data = results["MotChallenge2DBox"][tracker_name]["COMBINED_SEQ"]

    hota_val = res_data["HOTA"]["HOTA"]
    if isinstance(hota_val, np.ndarray) or isinstance(hota_val, list):
        hota = float(np.mean(hota_val)) # переводим в проценты, если нужно
    else:
        hota = float(hota_val) if float(hota_val) <= 1.0 else float(hota_val)

    mota = float(res_data["CLEAR"]["MOTA"])
    idf1 = float(res_data["Identity"]["IDF1"])

    if os.path.exists(tmp):
        shutil.rmtree(tmp)
    return {
        "MOTA": mota if mota is not None else 0.0,
        "IDF1": idf1 if idf1 is not None else 0.0,
        "HOTA": hota if hota is not None else 0.0,
    }




In [ ]:
from pprint import pprint

In [ ]:
def compute_metrics(sequence_path, pred_file):
    print(f"Evaluating sequence: {sequence_path}")
    seq_name = os.path.basename(sequence_path)

    tmp = tempfile.mkdtemp()
    gt_root = os.path.join(tmp, "gt")
    tracker_root = os.path.join(tmp, "trackers")

    benchmark = "MOT16"

    # Создаем структуру папок
    gt_seq_dir = os.path.join(gt_root, seq_name)
    os.makedirs(gt_seq_dir, exist_ok=True)

    # Копируем конфиг seqinfo.ini
    src_ini = os.path.join(sequence_path, "seqinfo.ini")
    dst_ini = os.path.join(gt_seq_dir, "seqinfo.ini")
    if os.path.exists(src_ini):
        shutil.copy(src_ini, dst_ini)

    # Копируем gt.txt
    gt_gt_dir = os.path.join(gt_seq_dir, "gt")
    os.makedirs(gt_gt_dir, exist_ok=True)
    shutil.copy(
        os.path.join(sequence_path, "gt", "gt.txt"),
        os.path.join(gt_gt_dir, "gt.txt")
    )

    tracker_name = "DeepSORT"
    tracker_dir = os.path.join(tracker_root, tracker_name, "data")
    os.makedirs(tracker_dir, exist_ok=True)
    shutil.copy(pred_file, os.path.join(tracker_dir, f"{seq_name}.txt"))

    # Генерируем seqmap
    seqmap_dir = os.path.join(tmp, "seqmaps")
    os.makedirs(seqmap_dir, exist_ok=True)
    seqmap_path = os.path.join(seqmap_dir, f"{benchmark}-train.txt")

    with open(seqmap_path, "w") as f:
        f.write("name\n")
        f.write(seq_name + "\n")

    # Настройки оценщика
    eval_config = Evaluator.get_default_eval_config()
    dataset_config = MotChallenge2DBox.get_default_dataset_config()

    dataset_config["GT_FOLDER"] = gt_root
    dataset_config["TRACKERS_FOLDER"] = tracker_root
    dataset_config["BENCHMARK"] = benchmark
    dataset_config["SKIP_SPLIT_FOL"] = True
    dataset_config["SEQMAP_FOLDER"] = seqmap_dir
    dataset_config["TRACKERS_TO_EVAL"] = [tracker_name]
    dataset_config["DO_PREPROC"] = False
    dataset_config["GT_LOC_FORMAT"] = "{gt_folder}/{seq}/gt/gt.txt"

    evaluator = Evaluator(eval_config)
    dataset = MotChallenge2DBox(dataset_config)

    metrics = [HOTA(), CLEAR(), Identity()]
    results, _ = evaluator.evaluate([dataset], metrics)
    pprint(results)
    print(results.keys())

    print(results["MotChallenge2DBox"].keys())

    print(results["MotChallenge2DBox"][tracker_name].keys())

    pprint(results["MotChallenge2DBox"][tracker_name]["COMBINED_SEQ"])

    # Извлекаем COMBINED_SEQ
    res_data = results["MotChallenge2DBox"][tracker_name]["COMBINED_SEQ"]

    # Приводим все ключи верхнего уровня к нижнему регистру для надежности
    res_data_lower = {k.lower(): v for k, v in res_data.items()}

    if os.path.exists(tmp):
        shutil.rmtree(tmp)
    return {
        "MOTA": mota,
        "IDF1": idf1,
        "HOTA": hota,
    }




In [ ]:
def run_experiment(video_path, sequence,
                   detector_name, reid_name,
                   conf, cosine_distance, nn_budget):

    # Гарантируем, что папка на Диске существует
    os.makedirs(RESULTS_DIR, exist_ok=True)

    sequence_name = os.path.basename(sequence)

    detector = create_detector(detector_name)
    detector.conf = conf

    reid = create_reid(reid_name)

    tracker = DeepSORTWrapper(
        max_cosine_distance=cosine_distance,
        nn_budget=nn_budget
    )

    # ИСПРАВЛЕНО: Теперь пути абсолютные и ведут на Google Диск
    output_video = os.path.join(
        VIDEOS_DIR,
        f"{sequence_name}_{detector_name}_{reid_name}.mp4"
    )

    output_mot = os.path.join(
        RESULTS_DIR,
        f"{sequence_name}_{detector_name}_{reid_name}.txt"
    )

    fps = run_sequence(
        img_dir=video_path,
        output_video=output_video,
        mot_path=output_mot,
        detector=detector,
        reid=reid,
        tracker=tracker
    )

    return {
        "fps": fps,
        "mot_file": output_mot
    }

In [ ]:
def run_cached(sequence, video_path,
               detector, reid,
               conf, cosine, budget):

    key = make_key(sequence, detector, reid, conf, cosine, budget)

    cached = load_cache(key)
    if cached:
        print("CACHE HIT:", sequence, detector, reid)
        return cached

    print("RUN:", sequence, detector, reid)

    result = run_experiment(
        video_path, sequence,
        detector, reid,
        conf, cosine, budget
    )

    save_cache(key, result)
    return result

In [ ]:
import shutil
# Удаляем старую папку с кэшем и создаем ее заново
if os.path.exists(CACHE_DIR):
    shutil.rmtree(CACHE_DIR)
os.makedirs(CACHE_DIR, exist_ok=True)
print("Кэш успешно очищен!")

Кэш успешно очищен!


In [ ]:
METRICS_DIR = "/content/drive/MyDrive/DeepSORT/metrics"
OUTPUT_CSV = os.path.join(METRICS_DIR, "experiments.csv")
file_exists = os.path.exists(OUTPUT_CSV)

processed_combinations = set()
if file_exists:
    try:
        df_existing = pd.read_csv(OUTPUT_CSV)
        # Собираем уникальный ключ для каждой строки, например, в виде кортежа параметров
        for _, r in df_existing.iterrows():
            # Приводим типы к единому стандарту, чтобы сравнение было точным
            key = (
                str(r["sequence"]),
                str(r["detector"]),
                str(r["reid"]),
                float(r["conf"]),
                float(r["cosine_distance"]),
                int(r["nn_budget"])
            )
            processed_combinations.add(key)
        print(f"Найдено {len(processed_combinations)} уже обработанных комбинаций. Они будут пропущены.")
    except Exception as e:
        print("Не удалось прочитать существующий csv для пропуска дубликатов, считаем всё с нуля:", e)

Найдено 2 уже обработанных комбинаций. Они будут пропущены.


In [ ]:

with open(OUTPUT_CSV, "a", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)

    if not file_exists:
        writer.writeheader()
        f.flush()
        print(123456789)

    for sequence in SEQUENCES:

        sequence_path = os.path.join(DATA_DIR, sequence)
        video_path = os.path.join(sequence_path, "img1")

        for detector in detectors:
            for reid in reids:
                for conf in confidences:
                    for cosine in cosine_distances:
                        for budget in nn_budgets:


                            exp = run_cached(
                                sequence_path,
                                video_path,
                                detector,
                                reid,
                                conf,
                                cosine,
                                budget
                            )

                            metrics = compute_metrics(sequence_path, exp["mot_file"])

                            row = {
                                "sequence": sequence,
                                "detector": detector,
                                "reid": reid,
                                "conf": conf,
                                "cosine_distance": cosine,
                                "nn_budget": budget,
                                "fps": exp["fps"],
                                **metrics
                            }

                            print(row)

                            writer.writerow(row)
                            f.flush()

                            print("OK:", sequence, detector, reid)

                            # except Exception as e:
                            #     print("FAILED:", sequence, detector, reid)
                            #     print(e)

RUN: /content/drive/MyDrive/DeepSORT/MOT_datasets/KITTI-17 yolov5n osnet_x0_25
PRO TIP 💡 Replace 'model=yolov5n.pt' with new 'model=yolov5nu.pt'.
YOLOv5 'u' models are trained with https://github.com/ultralytics/ultralytics and feature improved performance vs standard YOLOv5 models trained with https://github.com/ultralytics/yolov5.
Successfully loaded imagenet pretrained weights from "/root/.cache/torch/checkpoints/osnet_x0_25_imagenet.pth"
Frame 50/145 FPS=4.62
Frame 100/145 FPS=4.47
DONE
AVG FPS: 4.391248560337262
Evaluating sequence: /content/drive/MyDrive/DeepSORT/MOT_datasets/KITTI-17

Eval Config:
USE_PARALLEL         : False                         
NUM_PARALLEL_CORES   : 8                             
BREAK_ON_ERROR       : True                          
RETURN_ON_ERROR      : False                         
LOG_ON_ERROR         : /content/TrackEval/error_log.txt
PRINT_RESULTS        : True                          
PRINT_ONLY_COMBINED  : False                         
PRINT_CO

KeyboardInterrupt: 

<Figure size 640x480 with 0 Axes>